# Preprocesamiento de datos para el modelo

Este notebook prepara los datos para entrenar los modelos de predicción de `SalePrice`. Parte de las 30 variables seleccionadas en el análisis exploratorio (`pipeline_config.yaml`) y les aplica, en este orden:

1. Imputación de valores faltantes (numéricos y categóricos).
2. Codificación de variables categóricas.
3. Tratamiento de valores atípicos (outliers).
4. Transformación de variables continuas sesgadas.
5. Escalado de variables.

Cada paso se define en su propia celda y al final todos se unen en un **pipeline de scikit-learn**, construido con transformadores de **feature-engine**. El pipeline se ajusta solo con los datos de entrenamiento y se guarda junto con los datasets procesados, que serán el insumo del siguiente notebook (construcción de modelos).

## 1. Librerías

- **numpy** y **pandas**: manejo de datos.
- **yaml**: lectura del archivo de configuración con las variables seleccionadas.
- **scikit-learn**: división de los datos, codificación con orden definido, escalado y el `Pipeline` que encadena los pasos.
- **feature-engine**: transformadores de imputación, codificación, recorte de outliers y transformación que trabajan directamente sobre DataFrames de pandas y permiten indicar a qué columnas se aplica cada paso.
- **joblib**: guardado del pipeline ajustado para reutilizarlo después.

In [1]:
# Librerías
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder as SklearnOrdinalEncoder
from sklearn.preprocessing import StandardScaler

from feature_engine.encoding import OrdinalEncoder
from feature_engine.imputation import CategoricalImputer, MeanMedianImputer
from feature_engine.outliers import Winsorizer
from feature_engine.transformation import YeoJohnsonTransformer
from feature_engine.wrappers import SklearnTransformerWrapper

pd.set_option("display.max_columns", None)

## 2. Carga de datos y configuración

Se leen los datos originales de entrenamiento y prueba, y la lista de variables seleccionadas (`MODEL_FEATURES`) desde `pipeline_config.yaml`. Igual que en el notebook de exploración, las rutas no están fijas:

- La raíz del proyecto se detecta buscando el archivo `pyproject.toml`.
- La carpeta de datos es `data/raw/` por defecto y se puede cambiar con la variable de entorno `DATA_DIR`.

In [2]:
# Carga de datos y de la lista de variables seleccionadas
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA_DIR = Path(os.getenv("DATA_DIR", PROJECT_ROOT / "data" / "raw"))
CONFIG_FILE = PROJECT_ROOT / "pipeline_config.yaml"
TRAIN_FILE = "train.csv"
TEST_FILE = "test.csv"

TARGET = "SalePrice"
ID_COL = "Id"

config = yaml.safe_load(CONFIG_FILE.read_text(encoding="utf-8"))
MODEL_FEATURES = config["MODEL_FEATURES"]

train = pd.read_csv(DATA_DIR / TRAIN_FILE)
test = pd.read_csv(DATA_DIR / TEST_FILE)

print(f"Train: {train.shape} | Test: {test.shape} | Features seleccionadas: {len(MODEL_FEATURES)}")

Train: (1460, 81) | Test: (1459, 80) | Features seleccionadas: 30


## 3. Clasificación de las variables por tipo

Cada paso del preprocesamiento se aplica a un grupo distinto de variables, así que se clasifican las 30 features con la misma regla del análisis exploratorio:

- **Categóricas**: variables de texto.
- **Discretas**: numéricas con 15 valores distintos o menos.
- **Continuas**: el resto de las numéricas.

Además, las categóricas se dividen en dos subgrupos según cómo se codificarán:

- **Ordinales**: escalas con un orden natural. Por ejemplo, la calidad va de *Po* (pobre) a *Ex* (excelente). El orden de cada escala se toma de `data_description.txt`.
- **Nominales**: categorías sin un orden propio, como el barrio o el tipo de cimentación.

In [3]:
# Clasificación de las features en continuas, discretas y categóricas
MAX_DISCRETE_VALUES = 15

categoricas = [col for col in MODEL_FEATURES if not pd.api.types.is_numeric_dtype(train[col])]
numericas = [col for col in MODEL_FEATURES if col not in categoricas]
discretas = [col for col in numericas if train[col].nunique() <= MAX_DISCRETE_VALUES]
continuas = [col for col in numericas if col not in discretas]

# Escalas ordinales, de menor a mayor (según data_description.txt)
QUALITY_SCALE = ["Po", "Fa", "TA", "Gd", "Ex"]
ORDINAL_SCALES = {
    "ExterQual": QUALITY_SCALE,
    "BsmtQual": QUALITY_SCALE,
    "KitchenQual": QUALITY_SCALE,
    "FireplaceQu": QUALITY_SCALE,
    "HeatingQC": QUALITY_SCALE,
    "GarageFinish": ["Unf", "RFn", "Fin"],
    "BsmtFinType1": ["Unf", "LwQ", "Rec", "BLQ", "ALQ", "GLQ"],
}
ordinales = [col for col in categoricas if col in ORDINAL_SCALES]
nominales = [col for col in categoricas if col not in ORDINAL_SCALES]

print(f"Continuas ({len(continuas)}):   {continuas}")
print(f"Discretas ({len(discretas)}):   {discretas}")
print(f"Ordinales ({len(ordinales)}):   {ordinales}")
print(f"Nominales ({len(nominales)}):   {nominales}")

Continuas (10):   ['GrLivArea', 'GarageArea', 'TotalBsmtSF', '1stFlrSF', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt', 'MasVnrArea', 'BsmtFinSF1', 'LotFrontage']
Discretas (10):   ['OverallQual', 'GarageCars', 'FullBath', 'TotRmsAbvGrd', 'MSSubClass', 'Fireplaces', 'OverallCond', 'HalfBath', 'BedroomAbvGr', 'BsmtFullBath']
Ordinales (7):   ['ExterQual', 'BsmtQual', 'KitchenQual', 'GarageFinish', 'FireplaceQu', 'BsmtFinType1', 'HeatingQC']
Nominales (3):   ['Neighborhood', 'Foundation', 'GarageType']


## 4. División en entrenamiento y validación, y transformación del target

Se separa `train.csv` en dos partes:

- **Entrenamiento (80 %)**: con estos datos se ajusta el pipeline. Es decir, se calculan las medianas, las modas, los límites de outliers, etc.
- **Validación (20 %)**: se reserva para medir el desempeño de los modelos con datos que el pipeline no ha visto. Así se evita una evaluación demasiado optimista (fuga de información).

Se fija `random_state` para que la división sea siempre la misma.

El target `SalePrice` se transforma con **log(1 + x)** (`np.log1p`). Los precios tienen una cola larga hacia valores altos, y el logaritmo vuelve su distribución más simétrica y su relación con las variables más lineal. Las predicciones del modelo se regresan a dólares con `np.expm1`.

In [4]:
# División 80/20 y logaritmo del target
TEST_SIZE = 0.2
RANDOM_STATE = 42

X_train, X_val, y_train, y_val = train_test_split(
    train[MODEL_FEATURES], train[TARGET], test_size=TEST_SIZE, random_state=RANDOM_STATE
)
X_test = test[MODEL_FEATURES]

y_train_log = np.log1p(y_train)
y_val_log = np.log1p(y_val)

print(f"Entrenamiento: {X_train.shape} | Validación: {X_val.shape} | Test: {X_test.shape}")

Entrenamiento: (1168, 30) | Validación: (292, 30) | Test: (1459, 30)


## 5. Paso 1: Imputación de valores faltantes

Los modelos no aceptan celdas vacías, así que se rellenan:

- **Numéricas**: con la **mediana** de cada variable (`MeanMedianImputer`). La mediana no se ve afectada por los valores extremos ni por el sesgo, a diferencia del promedio. Se aplica a todas las numéricas, aunque algunas solo tengan vacíos en test (por ejemplo, `GarageArea`), para que el pipeline pueda procesar cualquier dato nuevo.
- **Categóricas**: con la **moda**, es decir, la categoría más frecuente de cada variable (`CategoricalImputer`).

Los valores de relleno se calculan solo con los datos de entrenamiento.

In [5]:
# Imputación: mediana para numéricas, moda para categóricas
imputer_numericas = MeanMedianImputer(imputation_method="median", variables=numericas)
imputer_categoricas = CategoricalImputer(imputation_method="frequent", variables=categoricas)

## 6. Paso 2: Codificación de variables categóricas

Los modelos trabajan con números, así que las categorías se convierten en valores numéricos. Cada variable sigue ocupando una sola columna:

- **Ordinales**: se respeta su orden natural. En las escalas de calidad, *Po* = 0, *Fa* = 1, *TA* = 2, *Gd* = 3 y *Ex* = 4, así que un número mayor significa mejor calidad. Si aparece una categoría que no está en la escala, recibe el valor -1.
- **Nominales**: como no tienen orden propio, se les asigna uno según el **precio promedio** de las viviendas de cada categoría en entrenamiento (`OrdinalEncoder` de feature-engine con `encoding_method="ordered"`). La categoría más barata recibe 0, la siguiente 1, y así sucesivamente. Si en datos nuevos aparece una categoría que no existía en entrenamiento, se codifica como -1.

In [6]:
# Codificación: orden natural para ordinales, orden por precio promedio para nominales
encoder_ordinales = SklearnTransformerWrapper(
    transformer=SklearnOrdinalEncoder(
        categories=[ORDINAL_SCALES[col] for col in ordinales],
        handle_unknown="use_encoded_value",
        unknown_value=-1,
    ),
    variables=ordinales,
)
encoder_nominales = OrdinalEncoder(encoding_method="ordered", variables=nominales, unseen="encode")

## 7. Paso 3: Tratamiento de outliers

Los valores extremos pueden distorsionar el modelo, así que se **recortan** (*capping*) en lugar de eliminar filas. Se usa `Winsorizer` con la regla del rango intercuartílico (IQR), la misma que definen los bigotes de un boxplot:

- Límite inferior: Q1 − 1.5 · IQR
- Límite superior: Q3 + 1.5 · IQR

Todo valor por debajo del límite inferior se reemplaza por ese límite, y lo mismo con el superior. Se aplica solo a las **variables continuas**, y los límites se calculan con los datos de entrenamiento.

In [7]:
# Recorte de outliers en variables continuas (regla IQR 1.5, ambas colas)
winsorizer = Winsorizer(capping_method="iqr", tail="both", fold=1.5, variables=continuas)

## 8. Paso 4: Transformación de variables continuas sesgadas

Algunas variables continuas tienen una cola larga hacia valores altos (asimetría). Esto hace que su relación con el precio sea menos lineal. Se transforman con **Yeo-Johnson** (`YeoJohnsonTransformer`), que busca automáticamente la transformación que vuelve cada variable lo más simétrica posible y, a diferencia del logaritmo, acepta ceros y valores negativos.

Solo se transforman las continuas con **|sesgo| > 0.75**, un umbral habitual para considerar que una distribución está claramente sesgada. El sesgo se mide en los datos de entrenamiento.

In [8]:
# Yeo-Johnson para continuas con |sesgo| > 0.75
SKEW_THRESHOLD = 0.75

skewness = X_train[continuas].skew()
continuas_sesgadas = skewness[skewness.abs() > SKEW_THRESHOLD].index.tolist()

yeo_johnson = YeoJohnsonTransformer(variables=continuas_sesgadas)

print("Continuas a transformar (sesgo en entrenamiento):")
print(skewness[continuas_sesgadas].round(2).sort_values(ascending=False).to_string())

Continuas a transformar (sesgo en entrenamiento):
LotFrontage    2.41
MasVnrArea     2.29
BsmtFinSF1     1.86
TotalBsmtSF    1.72
GrLivArea      1.43
1stFlrSF       1.42


## 9. Paso 5: Escalado de variables

Después de los pasos anteriores, las variables quedan en escalas muy distintas: por ejemplo, áreas en miles de pies² frente a calidades de 0 a 4. Para que ninguna domine solo por su escala, se aplica `StandardScaler` a **las 30 features**. Cada variable queda con media 0 y desviación estándar 1.

Se usa `SklearnTransformerWrapper` de feature-engine para que el resultado siga siendo un DataFrame con los nombres de las columnas.

In [9]:
# Escalado estándar de todas las features
scaler = SklearnTransformerWrapper(transformer=StandardScaler(), variables=MODEL_FEATURES)

## 10. Pipeline de preprocesamiento

Se unen los pasos anteriores en un único `Pipeline` de scikit-learn, en el orden en que deben aplicarse. Esto tiene tres ventajas:

- Con una sola instrucción (`fit`) se ajustan todos los pasos, siempre con los datos de entrenamiento.
- Con otra (`transform`) se aplican exactamente las mismas transformaciones a validación, test o cualquier dato nuevo.
- Se puede guardar y reutilizar tal cual en el notebook de modelos o en producción.

El pipeline se ajusta con `y_train_log` porque la codificación de las nominales necesita el precio para ordenar las categorías.

In [10]:
# Construcción y ajuste del pipeline
preprocessing_pipeline = Pipeline(steps=[
    ("imputer_numericas", imputer_numericas),
    ("imputer_categoricas", imputer_categoricas),
    ("encoder_ordinales", encoder_ordinales),
    ("encoder_nominales", encoder_nominales),
    ("winsorizer", winsorizer),
    ("yeo_johnson", yeo_johnson),
    ("scaler", scaler),
])

preprocessing_pipeline.fit(X_train, y_train_log)

X_train_proc = preprocessing_pipeline.transform(X_train)
X_val_proc = preprocessing_pipeline.transform(X_val)
X_test_proc = preprocessing_pipeline.transform(X_test)

preprocessing_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer_numericas', ...), ('imputer_categoricas', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0",list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,30
,variables,"['GrLivArea', 'GarageArea', ...]"
,imputation_method,'median'
Name,Type,Value
feature_names_in_,list,"['Gr...ea', 'Ga...ea', 'To...SF', '1s...SF', ...]"


## 11. Verificación del resultado

Antes de guardar se comprueba que el preprocesamiento funcionó:

- **Vacíos**: no debe quedar ninguno en entrenamiento, validación ni test.
- **Tipos**: todas las columnas deben ser numéricas.
- **Sesgo**: las continuas transformadas deben quedar con un sesgo cercano a 0.
- **Escala**: en entrenamiento, cada variable debe tener media ≈ 0 y desviación ≈ 1.

In [11]:
# Verificaciones de calidad del dataset procesado
for name, df in {"Entrenamiento": X_train_proc, "Validación": X_val_proc, "Test": X_test_proc}.items():
    print(f"{name:<14} forma: {df.shape} | vacíos: {df.isna().sum().sum()} | "
          f"todas numéricas: {all(pd.api.types.is_numeric_dtype(df[c]) for c in df.columns)}")

comparacion_sesgo = pd.DataFrame({
    "sesgo_original": X_train[continuas_sesgadas].skew(),
    "sesgo_procesado": X_train_proc[continuas_sesgadas].skew(),
}).round(2)
print("\nSesgo antes y después (entrenamiento):")
print(comparacion_sesgo.to_string())

print(f"\nMedia máx. |μ|: {X_train_proc.mean().abs().max():.3f} | Desviación: {X_train_proc.std().min():.3f} – {X_train_proc.std().max():.3f}")

Entrenamiento  forma: (1168, 30) | vacíos: 0 | todas numéricas: True
Validación     forma: (292, 30) | vacíos: 0 | todas numéricas: True
Test           forma: (1459, 30) | vacíos: 0 | todas numéricas: True

Sesgo antes y después (entrenamiento):
             sesgo_original  sesgo_procesado
GrLivArea              1.43            -0.01
TotalBsmtSF            1.72             0.05
1stFlrSF               1.42            -0.00
MasVnrArea             2.29             0.37
BsmtFinSF1             1.86            -0.42
LotFrontage            2.41             0.01

Media máx. |μ|: 0.000 | Desviación: 1.000 – 1.000


## 12. Guardado de los datasets procesados y del pipeline

Se guardan los resultados para el notebook de construcción de modelos:

| Archivo | Contenido |
|---|---|
| `data/processed/train_processed.csv` | `Id`, las 30 features procesadas y `SalePrice_log` (80 % de entrenamiento) |
| `data/processed/val_processed.csv` | `Id`, las 30 features procesadas y `SalePrice_log` (20 % de validación) |
| `data/processed/test_processed.csv` | `Id` y las 30 features procesadas (sin target; es el conjunto a predecir) |
| `models/preprocessing_pipeline.joblib` | El pipeline ajustado, para procesar datos nuevos de la misma forma |

La columna `Id` no es una feature: se guarda solo para identificar cada vivienda. El target se guarda ya en escala logarítmica (`SalePrice_log`).

In [12]:
# Guardado de datasets procesados y del pipeline ajustado
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
TARGET_LOG = f"{TARGET}_log"

train_out = X_train_proc.assign(**{TARGET_LOG: y_train_log})
val_out = X_val_proc.assign(**{TARGET_LOG: y_val_log})
train_out.insert(0, ID_COL, train.loc[X_train.index, ID_COL])
val_out.insert(0, ID_COL, train.loc[X_val.index, ID_COL])
test_out = X_test_proc.copy()
test_out.insert(0, ID_COL, test[ID_COL])

train_out.to_csv(PROCESSED_DIR / "train_processed.csv", index=False)
val_out.to_csv(PROCESSED_DIR / "val_processed.csv", index=False)
test_out.to_csv(PROCESSED_DIR / "test_processed.csv", index=False)
joblib.dump(preprocessing_pipeline, MODELS_DIR / "preprocessing_pipeline.joblib")

print(f"Datasets guardados en: {PROCESSED_DIR.relative_to(PROJECT_ROOT)}")
print(f"Pipeline guardado en:  {(MODELS_DIR / 'preprocessing_pipeline.joblib').relative_to(PROJECT_ROOT)}")

Datasets guardados en: data\processed
Pipeline guardado en:  models\preprocessing_pipeline.joblib
